# End-to-end validation

## Runtime contract

- **Inputs/runtime:** complete current-state Silver, Gold dimensions/facts and materialized lake views; Fabric Spark. `run_id` correlates validation, never filters to today's batch. BUDGET requires embedded explicit financeContext metadata.
- **Checks:** availability/non-empty state, exact complete source keys, duplicate grains, currency identity and deterministic keys, dimension/fiscal-period references, and financial reconciliation before checkpoints may commit.
- **Actual currencies:** HSL/RHCUR, KSL/RKCUR, TSL/persisted RTCUR, WSL/RWCUR reconcile separately at complete client/ledger/company/year/document/line/period keys plus each measure's currency. No currency is borrowed from Silver to conceal incorrect Gold identity.
- **Budget:** full current budget plus FAGLFLEXP periods 1-16 reconcile scalar-by-scalar at the previous nine-column grain extended with RCLNT/RLDNR/RHCUR. Explicit BUDGET_FILE client/ledger and unique as-of T001-derived WAERS are independently resolved; no currency-less total is constructed.
- **Compatibility:** legacy monetary names and precise decimal calculations remain. `AMOUNT_GROUP` is the legacy name for KSL/global currency RKCUR, not a universal group-currency guarantee. Calendar fiscal mapping and carry-forward 000 remain supported.
- **Outputs/failure:** existing append-only object evidence and bounded contract results; missing objects use catalog checks, reads are not swallowed, and failed contracts raise real pipeline errors after evidence is displayed.


In [ ]:
run_id = ""  # Correlation only: reconcile all current Silver rows.


In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable
from functools import reduce
from datetime import datetime, timezone
import uuid

if not isinstance(run_id, str):
    raise TypeError("run_id must be a string")
run_id = run_id.strip()
BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
SLV = f"{BRZ_SLV_LH}.slv"
QTN = f"{BRZ_SLV_LH}.quarantine"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
SOURCE_SYSTEM = "SAP_S4HANA"
BATCH_RUN_ID = run_id or str(uuid.uuid4())
LOAD_TIME_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.autoCompact.enabled", "true")
print(BATCH_RUN_ID, LOAD_TIME_UTC)


In [ ]:
ACDOCA_KEYS = ["RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN"]
BUDGET_KEYS = ["COMPANY_CODE", "GL_ACCOUNT", "COST_CENTER", "PROFIT_CENTER", "FISCAL_YEAR", "FISCAL_PERIOD", "PLAN_VERSION"]
LEGACY_BUDGET_GRAIN = [
    "PLAN_SOURCE", "SCENARIO", "COMPANY_CODE", "GL_ACCOUNT", "COST_CENTER",
    "PROFIT_CENTER", "FISCAL_YEAR", "FISCAL_PERIOD", "PLAN_VERSION",
]
BUDGET_SOURCE_GRAIN = LEGACY_BUDGET_GRAIN + ["RCLNT", "RLDNR"]
grain = BUDGET_SOURCE_GRAIN + ["RHCUR"]
BUDGET_MIGRATION_RECOVERY = (
    "No Gold fact changes were applied. Restore authoritative Silver/T001 SCD history for the legacy grain, "
    "or back up and explicitly rebuild fct.budget after approving its client/ledger/currency mapping. "
    "Do not append new keys beside unresolved legacy rows."
)
FINANCE_RECON_KEYS = ["RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN", "POPER"]
FINANCE_MEASURES = [
    ("local", "HSL", "AMOUNT_LOCAL", "RHCUR", "decimal(23,2)"),
    ("global", "KSL", "AMOUNT_GROUP", "RKCUR", "decimal(23,2)"),
    ("transaction", "TSL", "AMOUNT_TRANSACTION", "RTCUR", "decimal(23,2)"),
    ("other", "WSL", "AMOUNT_OTHER", "RWCUR", "decimal(23,2)"),
    ("quantity", "MSL", "QUANTITY", None, "decimal(23,3)"),
]


def key_subset(rows, keys_df, keys: list[str], how: str = "left_semi"):
    left, right = rows.alias("_rows"), keys_df.select(*keys).distinct().alias("_keys")
    match = reduce(lambda a, b: a & b, [
        F.col(f"_rows.{key}").eqNullSafe(F.col(f"_keys.{key}")) for key in keys
    ])
    return left.join(right, match, how)


def assert_checks(checks: list[tuple[str, bool, int]]) -> None:
    failed = [(name, failures) for name, passed, failures in checks if not passed]
    if failed:
        details = ", ".join(f"{name}={failures}" for name, failures in failed)
        raise RuntimeError(f"End-to-end validation failed: {details}")


def key_mismatch_count(expected, actual, keys: list[str]) -> int:
    return (key_subset(expected.select(*keys).distinct(), actual.select(*keys), keys, "left_anti").count()
            + key_subset(actual.select(*keys).distinct(), expected.select(*keys), keys, "left_anti").count())


def amount_mismatch_count(source, target, keys: list[str], source_amount: str, target_amount: str, data_type: str = "decimal(23,2)") -> int:
    def totals(rows, amount):
        return rows.groupBy(*keys).agg(F.count("*").alias("_ROWS"), F.count(F.col(amount)).alias("_AMOUNTS"),
                                       F.sum(F.col(amount).cast(data_type)).alias("_TOTAL"))
    left, right = totals(source, source_amount).alias("s"), totals(target, target_amount).alias("g")
    match = reduce(lambda a, b: a & b, [F.col(f"s.{key}").eqNullSafe(F.col(f"g.{key}")) for key in keys])
    mismatch = (~F.col("s._ROWS").eqNullSafe(F.col("g._ROWS"))
                | ~F.col("s._AMOUNTS").eqNullSafe(F.col("g._AMOUNTS"))
                | ~F.col("s._TOTAL").eqNullSafe(F.col("g._TOTAL")))
    return left.join(right, match, "full").filter(mismatch).count()


def row_amount_mismatch_count(source, target, keys: list[str], source_amount: str, target_amount: str, data_type: str = "decimal(23,2)") -> int:
    # Compare exact scalar amounts at the full client/ledger/currency grain;
    # no monetary total is allowed to combine unlike currencies.
    left = source.select(*keys, F.col(source_amount).cast(data_type).alias("_AMOUNT"), F.lit(True).alias("_PRESENT")).alias("s")
    right = target.select(*keys, F.col(target_amount).cast(data_type).alias("_AMOUNT"), F.lit(True).alias("_PRESENT")).alias("g")
    match = reduce(lambda a, b: a & b, [F.col(f"s.{key}").eqNullSafe(F.col(f"g.{key}")) for key in keys])
    mismatch = (~F.col("s._PRESENT").eqNullSafe(F.col("g._PRESENT"))
                | ~F.col("s._AMOUNT").eqNullSafe(F.col("g._AMOUNT")))
    return left.join(right, match, "full").filter(mismatch).count()











def business_key_hash(columns):
    return F.sha2(F.concat_ws("||", *[
        F.coalesce(column.cast("string"), F.lit("__NULL__")) for column in columns
    ]), 256)


def source_metadata(df):
    types = {"BATCH_RUN_ID": "string", "EXTRACT_DATE": "date", "LOAD_DATE": "date", "SOURCE_SYSTEM": "string", "SOURCE_FILE_NAME": "string"}
    return [(F.col(column).cast(data_type) if column in df.columns else F.lit(None).cast(data_type)).alias(column)
            for column, data_type in types.items()]


def budget_finance_context(config=None) -> tuple[str, str]:
    contract = globals().get("_SAP_EXTRACTION_CONFIG") if config is None else config
    if not isinstance(contract, dict) or not isinstance(contract.get("tables"), list):
        raise ValueError("BUDGET requires embedded _SAP_EXTRACTION_CONFIG.tables with explicit financeContext; no customer defaults are inferred")
    tables = [table for table in contract["tables"] if isinstance(table, dict) and str(table.get("name", "")).upper() == "BUDGET"]
    if len(tables) != 1 or not isinstance(tables[0].get("financeContext"), dict):
        raise ValueError("Exactly one BUDGET financeContext with explicit client and ledger is required")
    context = tables[0]["financeContext"]
    values = [context.get(key) for key in ["client", "ledger"]]
    if any(not isinstance(value, str) or not value.strip() for value in values):
        raise ValueError("BUDGET financeContext.client and .ledger must be non-empty strings; no inferred defaults are permitted")
    return values[0].strip(), values[1].strip()


def planning_date(year=None, period=None):
    year = F.col("FISCAL_YEAR") if year is None else year
    period = F.col("FISCAL_PERIOD") if period is None else period
    regular = F.make_date(year.cast("int"), F.greatest(F.lit(1), F.least(period.cast("int"), F.lit(12))), F.lit(1))
    return F.when(period.cast("int") <= 12, regular).otherwise(F.make_date(year.cast("int"), F.lit(12), F.lit(31)))


def assert_budget_unique(rows, keys: list[str], label: str):
    if rows.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise RuntimeError(f"{label}: ambiguous budget identity {keys}. {BUDGET_MIGRATION_RECOVERY}")


def resolve_budget_currency(rows):
    # dim.company is the T001-derived SCD2 mapping; do not use today's WAERS for historical plans.
    left = rows.withColumn("_PLAN_DATE", planning_date()).alias("b")
    company = spark.table(f"{DIM}.company").alias("c")
    required = {"MANDT", "BUKRS", "WAERS", "EFFECTIVE_FROM", "EFFECTIVE_TO"}
    if required - set(company.columns):
        raise ValueError("BUDGET currency requires the complete T001-derived company SCD2 mapping")
    match = (F.col("b.RCLNT").eqNullSafe(F.col("c.MANDT"))
             & F.col("b.COMPANY_CODE").eqNullSafe(F.col("c.BUKRS"))
             & F.col("b._PLAN_DATE").between(F.col("c.EFFECTIVE_FROM"), F.col("c.EFFECTIVE_TO")))
    resolved = left.join(company, match, "left").select("b.*", F.trim(F.col("c.WAERS").cast("string")).alias("RHCUR")).drop("_PLAN_DATE")
    assert_budget_unique(resolved, BUDGET_SOURCE_GRAIN, "BUDGET_FILE source/as-of T001 mapping")
    if resolved.filter(F.col("RHCUR").isNull() | (F.col("RHCUR") == "")).limit(1).count():
        raise ValueError("BUDGET_FILE has no unique populated as-of T001.WAERS for its explicit client/company/period; repair company history, never infer a currency")
    return resolved


def budget_projection(df, values: bool = True):
    client, ledger = budget_finance_context()
    columns = [F.col(column) for column in BUDGET_KEYS if column != "PLAN_VERSION"] + [
        F.coalesce(F.col("PLAN_VERSION"), F.lit("BUDGET")).alias("PLAN_VERSION"),
        F.lit("BUDGET_FILE").alias("PLAN_SOURCE"), F.lit("BUDGET").alias("SCENARIO"),
        F.lit(client).cast("string").alias("RCLNT"), F.lit(ledger).cast("string").alias("RLDNR"),
    ]
    if values:
        columns += [F.col("BUDGET_AMOUNT_LOCAL").cast("decimal(23,2)").alias("PLAN_AMOUNT_LOCAL"),
                    F.col("FORECAST_AMOUNT_LOCAL").cast("decimal(23,2)").alias("FORECAST_AMOUNT_LOCAL"),
                    *source_metadata(df), F.lit(client).alias("_CLIENT"), F.lit(None).cast("string").alias("_KOKRS")]
    projected = df.select(*columns)
    return resolve_budget_currency(projected) if values else projected


def plan_projection(df, values: bool = True):
    periods = [F.struct(
        F.lit(period).alias("FISCAL_PERIOD"),
        (F.col(f"TSL{period:02d}").cast("decimal(23,2)") if values else F.lit(None).cast("decimal(23,2)")).alias("PLAN_AMOUNT_LOCAL"),
    ) for period in range(1, 17)]
    expanded = df.withColumn("_PERIOD", F.explode(F.array(*periods)))
    columns = [F.col("RBUKRS").alias("COMPANY_CODE"), F.col("RACCT").alias("GL_ACCOUNT"),
               F.lit(None).cast("string").alias("COST_CENTER"), F.col("PRCTR").alias("PROFIT_CENTER"),
               F.col("RYEAR").alias("FISCAL_YEAR"), F.col("_PERIOD.FISCAL_PERIOD").alias("FISCAL_PERIOD"),
               F.col("VERSN").alias("PLAN_VERSION"), F.lit("FAGLFLEXP").alias("PLAN_SOURCE"), F.lit("PLAN").alias("SCENARIO"),
               F.col("RCLNT").cast("string").alias("RCLNT"), F.col("RLDNR").cast("string").alias("RLDNR")]
    if values:
        if "RHCUR" not in df.columns:
            raise ValueError("Synthetic FAGLFLEXP projection requires its declared RHCUR; no currency is inferred")
        columns += [F.col("_PERIOD.PLAN_AMOUNT_LOCAL").alias("PLAN_AMOUNT_LOCAL"),
                    F.lit(None).cast("decimal(23,2)").alias("FORECAST_AMOUNT_LOCAL"), *source_metadata(df),
                    F.col("RCLNT").cast("string").alias("_CLIENT"),
                    (F.col("KOKRS") if "KOKRS" in df.columns else F.lit(None).cast("string")).alias("_KOKRS"),
                    F.trim(F.col("RHCUR").cast("string")).alias("RHCUR")]
    projected = expanded.select(*columns)
    if values:
        missing_identity = reduce(lambda a, b: a | b, [F.col(key).isNull() | (F.trim(F.col(key)) == "") for key in ["RCLNT", "RLDNR", "RHCUR"]])
        if projected.filter(missing_identity).limit(1).count():
            raise ValueError("Synthetic FAGLFLEXP rows require populated RCLNT/RLDNR/RHCUR")
    return projected


objects = (
    [f"{SLV}.{name}" for name in ["acdoca", "bkpf", "t001", "csks", "cepc", "ska1", "skat", "lfa1", "kna1", "faglflexp", "budget"]]
    + [f"{DIM}.{name}" for name in ["company", "gl_account", "cost_center", "profit_center", "division", "vendor", "customer", "date", "fiscal_period"]]
    + [f"{FCT}.{name}" for name in ["finance_transaction", "budget"]]
    + [f"{GLD_LH}.mlv.{name}" for name in ["finance_actual_monthly", "vendor_spend_monthly", "gl_balance_monthly", "plan_vs_actual_monthly"]]
)
rows = []
for object_name in objects:
    exists = spark.catalog.tableExists(object_name)
    rows.append((object_name, spark.table(object_name).count() if exists else 0, exists, "" if exists else "Not found in catalog"))
result = spark.createDataFrame(rows, "object_name string, row_count long, exists boolean, error string").withColumn("batch_run_id", F.lit(BATCH_RUN_ID)).withColumn("check_time", F.lit(LOAD_TIME_UTC))
result.write.mode("append").format("delta").saveAsTable(f"{AUD}.end_to_end_validation")
display(result.orderBy("object_name"))
missing_objects = sum(not exists for _, _, exists, _ in rows)
empty_objects = sum(exists and count == 0 for _, count, exists, _ in rows)
checks = [("required_objects_exist", missing_objects == 0, missing_objects), ("required_objects_non_empty", empty_objects == 0, empty_objects)]
if missing_objects:
    # Dependent table reads cannot be meaningful when their required object is missing.
    display(spark.createDataFrame(checks, "check_name string, passed boolean, failure_count long"))
    assert_checks(checks)

acdoca = spark.table(f"{SLV}.acdoca")
fact = spark.table(f"{FCT}.finance_transaction")
budget_fact = spark.table(f"{FCT}.budget")
source_duplicates = acdoca.groupBy(*ACDOCA_KEYS).count().filter(F.col("count") > 1).count()
duplicate_fact_keys = fact.groupBy("FINANCE_TRANSACTION_SK").count().filter(F.col("count") > 1).count()
expected_fact_rows = acdoca.count()
actual_fact_rows = fact.count()
finance_key_mismatches = key_mismatch_count(acdoca, fact, ACDOCA_KEYS)
checks += [
    ("silver_acdoca_complete_key_unique", source_duplicates == 0, source_duplicates),
    ("finance_fact_key_unique", duplicate_fact_keys == 0, duplicate_fact_keys),
    ("finance_fact_reconciles_to_current_acdoca", expected_fact_rows == actual_fact_rows, abs(actual_fact_rows - expected_fact_rows)),
    ("finance_fact_complete_keys_match", finance_key_mismatches == 0, finance_key_mismatches),
]

# Validate persisted currency identities; never relabel TSL with RWCUR or
# borrow its currency from Silver to conceal an incorrect Gold RTCUR.
if "RTCUR" not in acdoca.columns or "RTCUR" not in fact.columns:
    raise RuntimeError("Persisted finance RTCUR is required; run Gold04 additive migration before validation")
if {"RCLNT", "RLDNR", "RHCUR"} - set(budget_fact.columns):
    raise RuntimeError("Persisted budget client/ledger/currency is required; run Gold05 additive migration before validation")
gold_currencies = fact
base_currency_grain = FINANCE_RECON_KEYS
for label, source_amount, target_amount, currency, data_type in FINANCE_MEASURES:
    group_keys = base_currency_grain + ([currency] if currency else [])
    mismatches = amount_mismatch_count(acdoca, gold_currencies, group_keys, source_amount, target_amount, data_type)
    checks.append((f"finance_{label}_amounts_reconcile", mismatches == 0, mismatches))
    if currency:
        missing_currency = acdoca.filter(F.col(source_amount).isNotNull() & (F.col(source_amount) != 0) & F.col(currency).isNull()).count()
        checks.append((f"finance_{label}_currency_populated", missing_currency == 0, missing_currency))

expected_budget = budget_projection(spark.table(f"{SLV}.budget")).unionByName(plan_projection(spark.table(f"{SLV}.faglflexp")))
budget_key_mismatches = key_mismatch_count(expected_budget, budget_fact, grain)
duplicate_budget_keys = budget_fact.groupBy("BUDGET_FACT_SK").count().filter(F.col("count") > 1).count()
duplicate_budget_source = expected_budget.groupBy(*grain).count().filter(F.col("count") > 1).count()
duplicate_budget_identity = (expected_budget.groupBy(*BUDGET_SOURCE_GRAIN).count().filter(F.col("count") > 1).count()
                             + budget_fact.groupBy(*BUDGET_SOURCE_GRAIN).count().filter(F.col("count") > 1).count())
budget_missing_identity = reduce(lambda a, b: a | b, [F.col(key).isNull() | (F.trim(F.col(key)) == "") for key in ["RCLNT", "RLDNR", "RHCUR"]])
budget_missing_identity_count = budget_fact.filter(budget_missing_identity).count()
budget_invalid_hash = budget_fact.filter(~F.col("BUDGET_FACT_SK").eqNullSafe(business_key_hash([F.col(key) for key in grain]))).count()
checks += [
    ("budget_fact_key_unique", duplicate_budget_keys == 0, duplicate_budget_keys),
    ("budget_source_projected_grain_unique", duplicate_budget_source == 0, duplicate_budget_source),
    ("budget_fact_complete_keys_match", budget_key_mismatches == 0, budget_key_mismatches),
    ("budget_source_identity_unique", duplicate_budget_identity == 0, duplicate_budget_identity),
    ("budget_currency_identity_populated", budget_missing_identity_count == 0, budget_missing_identity_count),
    ("budget_extended_key_hash_correct", budget_invalid_hash == 0, budget_invalid_hash),
]
for amount in ["PLAN_AMOUNT_LOCAL", "FORECAST_AMOUNT_LOCAL"]:
    mismatches = row_amount_mismatch_count(expected_budget, budget_fact, grain, amount, amount)
    checks.append((f"budget_{amount.lower()}_reconciles", mismatches == 0, mismatches))

mandatory = ["COMPANY_SK", "GL_ACCOUNT_SK", "COST_CENTER_SK", "PROFIT_CENTER_SK", "DIVISION_SK"]
unresolved_mandatory = sum(fact.filter(F.col(key).isNull()).count() for key in mandatory)
unresolved_optional = sum(fact.filter(F.col(natural_key).isNotNull() & F.col(key).isNull()).count() for natural_key, key in [("LIFNR", "VENDOR_SK"), ("KUNNR", "CUSTOMER_SK")])
checks += [("mandatory_dimension_keys_resolved", unresolved_mandatory == 0, unresolved_mandatory),
           ("populated_optional_dimension_keys_resolved", unresolved_optional == 0, unresolved_optional)]
dimension_keys = {"company": "COMPANY_SK", "gl_account": "GL_ACCOUNT_SK", "cost_center": "COST_CENTER_SK", "profit_center": "PROFIT_CENTER_SK", "division": "DIVISION_SK", "vendor": "VENDOR_SK", "customer": "CUSTOMER_SK"}
dangling_keys = 0
duplicate_dimension_keys = 0
for dimension_name, fact_key in dimension_keys.items():
    dimension = spark.table(f"{DIM}.{dimension_name}")
    duplicate_dimension_keys += dimension.groupBy("DIMENSION_SK").count().filter(F.col("count") > 1).count()
    valid_keys = dimension.select(F.col("DIMENSION_SK").alias(fact_key))  # Include expired versions referenced by history.
    for fact_table in [fact] + ([budget_fact] if fact_key in mandatory else []):
        dangling_keys += key_subset(fact_table.select(fact_key).filter(F.col(fact_key).isNotNull()).distinct(), valid_keys, [fact_key], "left_anti").count()
checks += [("dimension_surrogate_keys_unique", duplicate_dimension_keys == 0, duplicate_dimension_keys),
           ("historical_dimension_references_retained", dangling_keys == 0, dangling_keys)]

fiscal_period_dimension = spark.table(f"{DIM}.fiscal_period")
duplicate_fiscal_period_keys = fiscal_period_dimension.groupBy("FISCAL_PERIOD_KEY").count().filter(F.col("count") > 1).count()
fiscal_period_sources = [f"{FCT}.finance_transaction", f"{FCT}.budget"] + [f"{GLD_LH}.mlv.{name}" for name in ["finance_actual_monthly", "vendor_spend_monthly", "gl_balance_monthly", "plan_vs_actual_monthly"]]
unresolved_fiscal_period_keys = 0
for source_name in fiscal_period_sources:
    source_keys = spark.table(source_name).select(F.col("FISCAL_PERIOD_KEY").cast("int").alias("FISCAL_PERIOD_KEY"))
    unresolved_fiscal_period_keys += source_keys.filter(F.col("FISCAL_PERIOD_KEY").isNull()).count()
    unresolved_fiscal_period_keys += key_subset(source_keys.filter(F.col("FISCAL_PERIOD_KEY").isNotNull()), fiscal_period_dimension.select("FISCAL_PERIOD_KEY"), ["FISCAL_PERIOD_KEY"], "left_anti").count()
checks += [("fiscal_period_dimension_key_unique", duplicate_fiscal_period_keys == 0, duplicate_fiscal_period_keys),
           ("fiscal_period_keys_resolved", unresolved_fiscal_period_keys == 0, unresolved_fiscal_period_keys)]
check_result = spark.createDataFrame(checks, "check_name string, passed boolean, failure_count long")
display(check_result.orderBy("check_name"))
assert_checks(checks)
